# Manhattan estimate and embedding norm

For the same sampled 15-puzzle states, this notebook computes Manhattan distance and the encoder embedding's Euclidean norm. The chart groups norms into fixed-width bins, uses bar height for sample count, and bar color for average Manhattan distance. Run training first so the configured checkpoint is available.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from config import CONFIG_PATH, load_config
from fifteen_puzzle import PuzzleWalkDataset
from solve import load_model

In [2]:
config = load_config()
dataset_path = config["dataset"]["save_path"]

if dataset_path:
    dataset_path = Path(dataset_path)
    if not dataset_path.is_absolute():
        dataset_path = CONFIG_PATH.parent / dataset_path

if dataset_path and dataset_path.exists():
    with np.load(dataset_path) as saved:
        dataset = PuzzleWalkDataset(
            saved["states"],
            saved["moves"],
            saved["path_ids"],
            saved["steps"],
        )
else:
    dataset = PuzzleWalkDataset.generate(
        paths=config["walk"]["paths"],
        length=config["walk"]["T"],
        seed=config["seed"],
    )

sample_count = min(5000, len(dataset))
rng = np.random.default_rng(config["seed"] + 1)
sample_indices = rng.choice(len(dataset), size=sample_count, replace=False)
sampled_states = dataset.states[sample_indices]

In [3]:
def manhattan_distance(state):
    distance = 0
    for position, tile in enumerate(state):
        if tile == 0:
            continue
        goal_position = int(tile) - 1
        row, column = divmod(position, 4)
        goal_row, goal_column = divmod(goal_position, 4)
        distance += abs(row - goal_row) + abs(column - goal_column)
    return distance

estimated_distances = np.asarray(
    [manhattan_distance(state) for state in sampled_states],
    dtype=np.float32,
)
model = load_model(config)
device = next(model.parameters()).device
states_tensor = torch.as_tensor(sampled_states.astype(np.int64), device=device)

with torch.inference_mode():
    embeddings = model.encoder(states_tensor)
embedding_norms = torch.linalg.vector_norm(embeddings, ord=2, dim=1).cpu().numpy()

len(sampled_states), estimated_distances[:5], embedding_norms[:5]

(5000,
 array([31., 20., 29., 32., 19.], dtype=float32),
 array([4.4919147, 3.6985028, 3.2795374, 3.499377 , 4.6448526],
       dtype=float32))

In [4]:
embedding_matrix = embeddings.cpu().numpy()
slice_count = 8
slice_rng = np.random.default_rng(config["seed"] + 2)
directions = slice_rng.normal(size=(embedding_matrix.shape[1], slice_count))
directions /= np.linalg.norm(directions, axis=0, keepdims=True)
projected_slices = embedding_matrix @ directions
bin_count = 30
normal_x = np.linspace(-4, 4, 400)
normal_density = np.exp(-0.5 * normal_x**2) / np.sqrt(2 * np.pi)
projection_bins = []

for slice_index in range(slice_count):
    values = projected_slices[:, slice_index]
    lower = float(values.min())
    upper = float(values.max())
    if np.isclose(lower, upper):
        bin_edges = np.linspace(lower - 0.5, upper + 0.5, bin_count + 1)
    else:
        bin_edges = np.linspace(lower, upper, bin_count + 1)
    bin_ids = np.digitize(values, bin_edges[1:-1])
    counts = np.bincount(bin_ids, minlength=bin_count)
    distance_sums = np.bincount(
        bin_ids, weights=estimated_distances, minlength=bin_count
    )
    average_distances = np.divide(
        distance_sums,
        counts,
        out=np.zeros_like(distance_sums),
        where=counts > 0,
    )
    nonempty = counts > 0
    projection_bins.append(
        (values, bin_edges, counts, average_distances, nonempty)
    )

color_values = np.concatenate(
    [averages[nonempty] for _, _, _, averages, nonempty in projection_bins]
)
cmap = plt.colormaps["viridis"]
color_scale = plt.Normalize(vmin=color_values.min(), vmax=color_values.max())
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)

for slice_index, (ax, item) in enumerate(zip(axes.flat, projection_bins)):
    values, bin_edges, counts, average_distances, nonempty = item
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_widths = np.diff(bin_edges)
    bar_colors = cmap(color_scale(average_distances[nonempty]))
    bars = ax.bar(
        bin_centers[nonempty],
        counts[nonempty],
        width=bin_widths[nonempty] * 0.9,
        color=bar_colors,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.bar_label(
        bars,
        labels=[str(int(count)) for count in counts[nonempty]],
        padding=1,
        fontsize=6,
    )
    expected_counts = normal_density * len(values) * bin_widths.mean()
    ax.plot(normal_x, expected_counts, color="black", linewidth=1)
    ax.set_title(f"Random slice {slice_index + 1}")
    ax.set_xlabel("Projection value")
    ax.grid(axis="y", alpha=0.25)

axes[0, 0].set_ylabel("Number of samples")
axes[1, 0].set_ylabel("Number of samples")
fig.suptitle("Random projection distributions colored by average Manhattan distance")
colorbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=color_scale, cmap=cmap),
    ax=axes.ravel().tolist(),
)
colorbar.set_label("Average Manhattan distance")
fig.tight_layout()
plt.show()

NameError: name 'projected_slices' is not defined